In [ ]:
import os, random, time, math
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision.models import resnet18


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [ ]:
DATA_ROOT = "/kaggle/input"

RVF_ROOT = os.path.join(DATA_ROOT, "140k-real-and-fake-faces", "real_vs_fake", "real-vs-fake")

FFHQ_ROOT = os.path.join(DATA_ROOT, "ffhq-1024x1024", "images1024x1024")

DEEP_ROOT = os.path.join(DATA_ROOT, "deepdetect-2025", "ddata")


In [ ]:
def assert_exists(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing path: {p}")
    return True

assert_exists(RVF_ROOT)
assert_exists(FFHQ_ROOT)
assert_exists(DEEP_ROOT)
print("Roots found")


In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")

def list_images(folder: str) -> List[str]:
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(IMG_EXTS):
                out.append(os.path.join(root, f))
    return out

def split_list(items: List[str], valid_ratio=0.1, seed=42):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)
    cut = int(len(items) * (1 - valid_ratio))
    train_idx, valid_idx = idx[:cut], idx[cut:]
    train_items = [items[i] for i in train_idx]
    valid_items = [items[i] for i in valid_idx]
    return train_items, valid_items


In [ ]:
def get_rvf_split(split_name: str):
    real_dir = os.path.join(RVF_ROOT, split_name, "real")
    fake_dir = os.path.join(RVF_ROOT, split_name, "fake")
    assert_exists(real_dir); assert_exists(fake_dir)
    return list_images(real_dir), list_images(fake_dir)

rvf_train_real, rvf_train_fake = get_rvf_split("train")
rvf_valid_real, rvf_valid_fake = get_rvf_split("valid")

print("RVF train real/fake:", len(rvf_train_real), len(rvf_train_fake))
print("RVF valid real/fake:", len(rvf_valid_real), len(rvf_valid_fake))


In [ ]:
ffhq_all = list_images(FFHQ_ROOT)
print("FFHQ 1024 total:", len(ffhq_all))

ffhq_train, ffhq_valid = split_list(ffhq_all, valid_ratio=0.1, seed=42)
print("FFHQ train/valid:", len(ffhq_train), len(ffhq_valid))


In [ ]:
deep_fake_train_dir = os.path.join(DEEP_ROOT, "train", "fake")
deep_fake_valid_dir = os.path.join(DEEP_ROOT, "test", "fake")  

assert_exists(deep_fake_train_dir)
assert_exists(deep_fake_valid_dir)

deep_fake_train = list_images(deep_fake_train_dir)
deep_fake_valid = list_images(deep_fake_valid_dir)

print("DeepDetect fake train:", len(deep_fake_train))
print("DeepDetect fake valid (from test):", len(deep_fake_valid))


In [ ]:
def rgb_to_luma_np(img: Image.Image) -> np.ndarray:
    arr = np.asarray(img).astype(np.float32) / 255.0
    y = 0.299 * arr[...,0] + 0.587 * arr[...,1] + 0.114 * arr[...,2]
    return y

def log_fft_mag(y: np.ndarray) -> np.ndarray:
    f = np.fft.fft2(y)
    fshift = np.fft.fftshift(f)
    mag = np.log1p(np.abs(fshift)).astype(np.float32)
    return mag

def ring_normalize(mag: np.ndarray, num_bins: int = 64) -> np.ndarray:
    h, w = mag.shape
    cy, cx = h // 2, w // 2

    yy, xx = np.ogrid[:h, :w]
    rr = np.sqrt((yy - cy)**2 + (xx - cx)**2)
    rmax = rr.max() + 1e-6

    bins = np.linspace(0, rmax, num_bins + 1, dtype=np.float32)
    out = mag.copy()

    for i in range(num_bins):
        mask = (rr >= bins[i]) & (rr < bins[i+1])
        if not np.any(mask):
            continue
        vals = mag[mask]
        mu = vals.mean()
        sd = vals.std() + 1e-6
        out[mask] = (vals - mu) / sd

    out = np.clip(out, -5.0, 5.0)
    return out.astype(np.float32)

def center_crop(img: Image.Image, size: int) -> Image.Image:
    w, h = img.size
    if w < size or h < size:
        scale = size / min(w, h)
        img = img.resize((int(w*scale)+1, int(h*scale)+1), resample=Image.BILINEAR)
        w, h = img.size
    left = (w - size) // 2
    top = (h - size) // 2
    return img.crop((left, top, left+size, top+size))

def random_crop(img: Image.Image, size: int) -> Image.Image:
    w, h = img.size
    if w < size or h < size:
        scale = size / min(w, h)
        img = img.resize((int(w*scale)+1, int(h*scale)+1), resample=Image.BILINEAR)
        w, h = img.size
    left = random.randint(0, w - size)
    top = random.randint(0, h - size)
    return img.crop((left, top, left+size, top+size))

def fft_multiscale_ringnorm_tensor(img: Image.Image, out_size=224, train=True) -> torch.Tensor:
    scales = [299, 224, 160]
    chans = []
    for s in scales:
        crop = random_crop(img, s) if train else center_crop(img, s)
        crop = crop.resize((out_size, out_size), resample=Image.BILINEAR)
        y = rgb_to_luma_np(crop)
        mag = log_fft_mag(y)
        mag = ring_normalize(mag, num_bins=64)
        chans.append(mag)
    stacked = np.stack(chans, axis=0) 
    return torch.from_numpy(stacked).float()


In [ ]:
from PIL import ImageFilter, ImageEnhance
from io import BytesIO

def jpeg_recompress(img: Image.Image, quality: int) -> Image.Image:
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def degraded_pipeline(img: Image.Image) -> Image.Image:
    if random.random() < 0.7:
        w, h = img.size
        scale = random.uniform(0.35, 0.95)
        nw, nh = max(64, int(w*scale)), max(64, int(h*scale))
        img = img.resize((nw, nh), resample=Image.BILINEAR).resize((w, h), resample=Image.BILINEAR)

    if random.random() < 0.8:
        img = jpeg_recompress(img, random.randint(25, 95))

    r = random.random()
    if r < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.1)))
    elif r < 0.35:
        img = img.filter(ImageFilter.UnsharpMask(radius=1, percent=random.randint(50, 150), threshold=3))

    if random.random() < 0.3:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.3:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))

    return img


In [ ]:
DOMAINS = {
    "real_old": 0,
    "real_hq": 1,
    "fake_gan": 2,
    "fake_modern": 3,
}
ID2DOMAIN = {v:k for k,v in DOMAINS.items()}


In [ ]:
class FFTDomainDataset(Dataset):
    def __init__(self, samples: List[Tuple[str,int,int]], train=True, out_size=224):
        self.samples = samples
        self.train = train
        self.out_size = out_size

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label, domain_id = self.samples[idx]
        img = Image.open(path).convert("RGB")

        if self.train:
            if random.random() < 0.30:
                img = degraded_pipeline(img)

        x = fft_multiscale_ringnorm_tensor(img, out_size=self.out_size, train=self.train)
        y = torch.tensor(label, dtype=torch.long)
        d = torch.tensor(domain_id, dtype=torch.long)
        return x, y, d


In [ ]:
def take_n(items: List[str], n: int, seed=42) -> List[str]:
    rng = np.random.RandomState(seed)
    if len(items) <= n:
        return items
    idx = rng.choice(len(items), size=n, replace=False)
    return [items[i] for i in idx]

N_REAL_OLD  = 50000   
N_REAL_HQ   = 50000   
N_FAKE_GAN  = 50000   
N_FAKE_MOD  = min(50000, len(deep_fake_train)) 

train_real_old = take_n(rvf_train_real, N_REAL_OLD, seed=1)
train_real_hq  = take_n(ffhq_train,    N_REAL_HQ,  seed=2)
train_fake_gan = take_n(rvf_train_fake, N_FAKE_GAN, seed=3)
train_fake_mod = take_n(deep_fake_train, N_FAKE_MOD, seed=4)

V_REAL_OLD  = min(10000, len(rvf_valid_real))
V_REAL_HQ   = min(10000, len(ffhq_valid))
V_FAKE_GAN  = min(10000, len(rvf_valid_fake))
V_FAKE_MOD  = min(10000, len(deep_fake_valid))

valid_real_old = take_n(rvf_valid_real, V_REAL_OLD, seed=11)
valid_real_hq  = take_n(ffhq_valid,     V_REAL_HQ,  seed=12)
valid_fake_gan = take_n(rvf_valid_fake, V_FAKE_GAN, seed=13)
valid_fake_mod = take_n(deep_fake_valid, V_FAKE_MOD, seed=14)

train_samples = []
train_samples += [(p, 0, DOMAINS["real_old"]) for p in train_real_old]
train_samples += [(p, 0, DOMAINS["real_hq"]) for p in train_real_hq]
train_samples += [(p, 1, DOMAINS["fake_gan"]) for p in train_fake_gan]
train_samples += [(p, 1, DOMAINS["fake_modern"]) for p in train_fake_mod]

valid_samples = []
valid_samples += [(p, 0, DOMAINS["real_old"]) for p in valid_real_old]
valid_samples += [(p, 0, DOMAINS["real_hq"]) for p in valid_real_hq]
valid_samples += [(p, 1, DOMAINS["fake_gan"]) for p in valid_fake_gan]
valid_samples += [(p, 1, DOMAINS["fake_modern"]) for p in valid_fake_mod]

print("Train samples:", len(train_samples))
print("Valid samples:", len(valid_samples))
print("Train domain sizes:",
      sum(1 for _,_,d in train_samples if d==DOMAINS["real_old"]),
      sum(1 for _,_,d in train_samples if d==DOMAINS["real_hq"]),
      sum(1 for _,_,d in train_samples if d==DOMAINS["fake_gan"]),
      sum(1 for _,_,d in train_samples if d==DOMAINS["fake_modern"]))


In [ ]:
def make_domain_balanced_sampler(samples):
    doms = [d for _,_,d in samples]
    counts = np.bincount(doms, minlength=len(DOMAINS))
    weights = np.array([1.0 / (counts[d] + 1e-6) for d in doms], dtype=np.float32)
    return WeightedRandomSampler(torch.from_numpy(weights), num_samples=len(samples), replacement=True)

train_ds = FFTDomainDataset(train_samples, train=True, out_size=224)
valid_ds = FFTDomainDataset(valid_samples, train=False, out_size=224)

train_sampler = make_domain_balanced_sampler(train_samples)

BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
class FFTResNetEmbed(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        base = resnet18(weights=None)
        in_feats = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.embed = nn.Linear(in_feats, emb_dim)
        self.classifier = nn.Linear(emb_dim, 2)

    def forward(self, x):
        feats = self.backbone(x)
        z = F.normalize(self.embed(feats), dim=1)
        logits = self.classifier(z)
        return logits, z


In [ ]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.08):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, target):
        n = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true = torch.zeros_like(logp)
            true.fill_(self.smoothing / (n - 1))
            true.scatter_(1, target.unsqueeze(1), 1 - self.smoothing)
        return torch.mean(torch.sum(-true * logp, dim=1))


In [ ]:
def supervised_contrastive_loss(z, y, temperature=0.2):
    B = z.size(0)
    sim = torch.matmul(z, z.T) / temperature  
    sim = sim - torch.max(sim, dim=1, keepdim=True)[0]  

    y = y.view(-1, 1)
    mask = (y == y.T).float().to(z.device)  

    logits_mask = torch.ones_like(mask) - torch.eye(B, device=z.device)
    mask = mask * logits_mask

    exp_sim = torch.exp(sim) * logits_mask
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)


    mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
    loss = -mean_log_prob_pos.mean()
    return loss


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0

    bucket_total = {k:0 for k in DOMAINS}
    bucket_correct = {k:0 for k in DOMAINS}

    hq_real_total = 0
    hq_real_correct = 0

    for x, y, d in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        d_np = d.cpu().numpy()

        logits, _ = model(x)
        pred = logits.argmax(dim=1)

        total += y.size(0)
        correct += (pred == y).sum().item()

        pred_np = pred.cpu().numpy()
        y_np = y.cpu().numpy()

        for i in range(len(d_np)):
            dom_name = ID2DOMAIN[int(d_np[i])]
            bucket_total[dom_name] += 1
            bucket_correct[dom_name] += int(pred_np[i] == y_np[i])

            if dom_name == "real_hq":
                hq_real_total += 1
                hq_real_correct += int(pred_np[i] == 0)  

    overall_acc = correct / max(1, total)
    bucket_acc = {f"acc_{k}": bucket_correct[k] / max(1, bucket_total[k]) for k in bucket_total}
    worst_bucket = min(bucket_acc.values()) if bucket_acc else overall_acc

    hq_real_recall = hq_real_correct / max(1, hq_real_total)

    out = {"acc_overall": overall_acc, "acc_worst_bucket": worst_bucket, "hq_real_recall": hq_real_recall}
    out.update(bucket_acc)
    return out


In [ ]:
model = FFTResNetEmbed(emb_dim=256).to(DEVICE)

ce_loss = LabelSmoothingCE(smoothing=0.08)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

EPOCHS = 6
LAMBDA_CONTRAST = 0.3  

best_score = -1.0
best_path = "best_freq_branch_v4.pt"

def train_one_epoch(model, loader):
    model.train()
    running = 0.0
    n = 0

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits, z = model(x)

        loss_ce = ce_loss(logits, y)
        loss_con = supervised_contrastive_loss(z, y, temperature=0.2)

        loss = loss_ce + LAMBDA_CONTRAST * loss_con

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running += loss.item() * x.size(0)
        n += x.size(0)

    return running / max(1, n)

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader)
    metrics = evaluate(model, valid_loader)
    scheduler.step()

    print(f"\nEpoch {epoch}/{EPOCHS} | loss={train_loss:.4f} | time={(time.time()-t0):.1f}s")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")


    score = 0.6 * metrics["hq_real_recall"] + 0.4 * metrics["acc_worst_bucket"]
    if score > best_score:
        best_score = score
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "score": score,
        }, best_path)
        print(f"✅ Saved best: score={best_score:.4f} -> {best_path}")

print("\nDone. Best score:", best_score)


In [23]:
print("plug")

plug
